# LAMU 2026 - Mapa pytan dzieci

### Czym jest ten notatnik?

Ten notatnik analizuje pytania zadane przez dzieci w ramach **LAMU 2026** (Letnia Akademia Mlodych Umyslow, projekt Radia Naukowego). Pytania zostaly zebrane w pliku PDF i pogrupowane w kategorie tematyczne (Fizyka, Biologia, Wszechswiat, itd.).

Notatnik robi cos, czego nie da sie zrobic recznie - bierze kazde pytanie i za pomoca **modelu jezykowego** zamienia je w wektor liczbowy (tzw. embedding), ktory "rozumie" znaczenie pytania. Dzieki temu mozna zobaczyc, ktore pytania sa do siebie podobne tresciowo, nawet jesli uzywaja roznych slow. Wynikiem jest **interaktywna mapa**, na ktorej kazde pytanie to punkt, a bliskosc punktow oznacza podobienstwo znaczeniowe.

### Jak uruchomic ten notatnik?

1. Otworz strone [Google Colab](https://colab.research.google.com/)
2. Wybierz **Plik > Otworz notatnik > Przeslij** i wgraj ten plik (.ipynb)
3. W menu u gory wybierz **Srodowisko uruchomieniowe > Zmien typ srodowiska** i ustaw **GPU** (typ T4) - dzieki temu obliczenia beda szybsze, ale notatnik zadziala tez bez GPU
4. Kliknij **Srodowisko uruchomieniowe > Uruchom wszystko** (lub Ctrl+F9)
5. Poczekaj kilka minut - notatnik sam pobierze PDF, zainstaluje potrzebne biblioteki, zakoduje pytania i wygeneruje wykresy
6. Na koncu automatycznie pobierze sie plik HTML z interaktywnym wykresem, ktory mozna otworzyc w przegladarce offline

### Co robi kazdy krok?

| Krok | Co sie dzieje | Ile trwa |
|------|--------------|----------|
| 1. Instalacja bibliotek | Pobiera narzedzia potrzebne do analizy | ~1 min |
| 2. Pobranie PDF | Sciaga plik z pytaniami ze strony Radia Naukowego | kilka sekund |
| 3. Ekstrakcja pytan | Wyciaga pytania z PDF i przypisuje im kategorie | natychmiastowe |
| 4. Embeddingi | Model jezykowy zamienia kazde pytanie w wektor 768 liczb | ~1-2 min (GPU) |
| 5. Redukcja wymiarow | Algorytm UMAP sciaga 768 wymiarow do 2 (zeby mozna bylo narysowac) | ~30 sek |
| 6. Wykresy | Rysuje statyczny (matplotlib) i interaktywny (Plotly) wykres | natychmiastowe |
| 7. Eksport HTML | Zapisuje interaktywny wykres jako plik do pobrania | natychmiastowe |

### Czego potrzeba?

Niczego poza przegladarka internetowa. Caly kod uruchamia sie w chmurze Google (Colab), nie trzeba nic instalowac na swoim komputerze. Konto Google jest wymagane do korzystania z Colab.

In [ ]:
!pip install -q sentence-transformers umap-learn plotly pymupdf requests

In [ ]:
import fitz  # pymupdf
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import umap
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'colab'

## 1. Pobranie i wczytanie pytan z PDF

In [ ]:
import requests

PDF_URL = "https://radionaukowe.pl/wp-content/uploads/2026/03/LAMU-zadane-pytania-2026.pdf"
PDF_PATH = "LAMU-zadane-pytania-2026.pdf"

response = requests.get(PDF_URL)
response.raise_for_status()
with open(PDF_PATH, "wb") as f:
    f.write(response.content)
print(f"Pobrano PDF: {PDF_PATH} ({len(response.content) / 1024:.0f} KB)")

In [ ]:
def extract_questions_from_pdf(pdf_path):
    """Extract questions grouped by topic from the LAMU PDF."""
    doc = fitz.open(pdf_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    doc.close()

    # Known topic headers in the PDF
    topics = ["FIZYKA", "BIOLOGIA", "WSZECHSWIAT", "CZLOWIEK",
              "TECHNOLOGIE", "ZIEMIA", "MATEMATYKA", "HISTORIA", "CHEMIA"]

    # Normalize text: replace special Polish chars in topic headers for matching
    # The PDF may have WSZECHŚWIAT and CZŁOWIEK with diacritics
    topic_pattern = r'(FIZYKA|BIOLOGIA|WSZECH[ŚS]WIAT|CZ[ŁL]OWIEK|TECHNOLOGIE|ZIEMIA|MATEMATYKA|HISTORIA|CHEMIA)'

    # Split text by topic headers
    parts = re.split(topic_pattern, full_text)

    questions = []
    current_topic = None

    for part in parts:
        part_stripped = part.strip()
        if re.match(topic_pattern, part_stripped):
            # Normalize topic name
            current_topic = part_stripped.replace('Ś', 'S').replace('Ł', 'L')
            # Map back to nice names with diacritics for display
            topic_display = {
                'FIZYKA': 'Fizyka',
                'BIOLOGIA': 'Biologia',
                'WSZECHSWIAT': 'Wszechświat',
                'CZLOWIEK': 'Człowiek',
                'TECHNOLOGIE': 'Technologie',
                'ZIEMIA': 'Ziemia',
                'MATEMATYKA': 'Matematyka',
                'HISTORIA': 'Historia',
                'CHEMIA': 'Chemia'
            }
            current_topic = topic_display.get(current_topic, current_topic)
            continue

        if current_topic is None:
            continue

        # Extract lines starting with '–' or '-'
        lines = part.split('\n')
        buffer = ""
        for line in lines:
            line = line.strip()
            if not line or line.startswith('LAMU') or line.startswith('Radio') or 'RADIO' in line.upper() or 'NAUKOWE' in line.upper() or 'LETNIA' in line.upper() or 'AKADEMIA' in line.upper():
                continue
            # Check if line starts with a dash (new question)
            if re.match(r'^[–\-]\s', line):
                if buffer:
                    questions.append((current_topic, buffer.strip()))
                buffer = re.sub(r'^[–\-]\s*', '', line)
            else:
                # Continuation of previous question
                if buffer:
                    buffer += ' ' + line

        if buffer:
            questions.append((current_topic, buffer.strip()))

    return questions

questions_data = extract_questions_from_pdf(PDF_PATH)
df = pd.DataFrame(questions_data, columns=['topic', 'question'])
print(f"Liczba pytan: {len(df)}")
print(f"\nPytania wg kategorii:")
print(df['topic'].value_counts())
print(f"\nPrzykladowe pytania:")
df.head(10)

## 2. Kodowanie pytan za pomoca polskich embeddingów

In [ ]:
MODEL_NAME = "sdadas/st-polish-paraphrase-from-distilroberta"

model = SentenceTransformer(MODEL_NAME)
print(f"Model: {MODEL_NAME}")
print(f"Wymiar embeddingów: {model.get_sentence_embedding_dimension()}")

In [ ]:
embeddings = model.encode(df['question'].tolist(), show_progress_bar=True, batch_size=32)
print(f"Ksztalt macierzy embeddingów: {embeddings.shape}")

## 3. Redukcja wymiarowości (UMAP)

In [ ]:
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)

coords_2d = reducer.fit_transform(embeddings)
df['x'] = coords_2d[:, 0]
df['y'] = coords_2d[:, 1]
print(f"Redukcja wymiarów: {embeddings.shape[1]} -> 2")

## 4. Wizualizacja

### 4a. Wykres statyczny (matplotlib)

In [ ]:
# Color palette for topics
topic_colors = {
    'Fizyka': '#1f77b4',
    'Biologia': '#2ca02c',
    'Wszechświat': '#9467bd',
    'Człowiek': '#d62728',
    'Technologie': '#ff7f0e',
    'Ziemia': '#8c564b',
    'Matematyka': '#e377c2',
    'Historia': '#7f7f7f',
    'Chemia': '#17becf'
}

fig, ax = plt.subplots(figsize=(14, 10))

for topic in df['topic'].unique():
    mask = df['topic'] == topic
    ax.scatter(
        df.loc[mask, 'x'], df.loc[mask, 'y'],
        label=topic,
        color=topic_colors.get(topic, '#333333'),
        s=50, alpha=0.7, edgecolors='white', linewidth=0.5
    )

ax.legend(title='Kategoria', fontsize=11, title_fontsize=12,
          loc='best', framealpha=0.9)
ax.set_title('LAMU 2026 - Pytania dzieci w przestrzeni embeddingów\n'
             f'(model: {MODEL_NAME}, redukcja: UMAP)', fontsize=14)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('lamu_questions_static.png', dpi=150, bbox_inches='tight')
plt.show()
print("Zapisano: lamu_questions_static.png")

### 4b. Wykres interaktywny (Plotly)

In [ ]:
# Truncate long questions for hover display
df['question_short'] = df['question'].apply(
    lambda q: q[:80] + '...' if len(q) > 80 else q
)

fig = px.scatter(
    df, x='x', y='y',
    color='topic',
    hover_data={'question': True, 'topic': True, 'x': False, 'y': False,
                'question_short': False},
    color_discrete_map=topic_colors,
    title='LAMU 2026 - Pytania dzieci (interaktywna mapa embeddingów)',
    labels={'topic': 'Kategoria', 'x': 'UMAP 1', 'y': 'UMAP 2'},
    width=900, height=700
)

fig.update_traces(
    marker=dict(size=8, opacity=0.75, line=dict(width=0.5, color='white')),
    hovertemplate='<b>%{customdata[1]}</b><br>%{customdata[0]}<extra></extra>'
)

fig.update_layout(
    legend_title_text='Kategoria',
    font=dict(size=12),
    hoverlabel=dict(font_size=11),
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgray'),
    yaxis=dict(showgrid=True, gridcolor='lightgray')
)

fig.show()

### 4c. Statystyki

In [ ]:
print("=" * 50)
print("LAMU 2026 - Podsumowanie")
print("=" * 50)
print(f"Calkowita liczba pytan: {len(df)}")
print(f"Liczba kategorii: {df['topic'].nunique()}")
print(f"\nPytania wg kategorii:")
for topic, count in df['topic'].value_counts().items():
    print(f"  {topic}: {count}")
print(f"\nModel embeddingów: {MODEL_NAME}")
print(f"Wymiar embeddingów: {embeddings.shape[1]}")
print(f"Metoda redukcji: UMAP (n_neighbors=15, min_dist=0.1, metric=cosine)")

### 4d. Dla organizatorow LAMU - jak czytac ten wykres i co z niego wynika

---

#### Jak korzystac z interaktywnego wykresu (Plotly)

Kazdy **punkt na wykresie to jedno pytanie** zadane przez dziecko. Kolor oznacza kategorie tematyczna (Fizyka, Biologia, itd.), zgodnie z legenda po prawej stronie.

**Najwazniejsza zasada: im blizej siebie sa dwa punkty, tym bardziej podobne sa pytania** - nie w sensie literalnym (te same slowa), lecz w sensie znaczeniowym. Model jezykowy "rozumie" tresc pytania i umieszcza je w przestrzeni tak, ze pytania o podobnych rzeczach laduja obok siebie.

Interakcje z wykresem:
- **Najechanie myszka** na punkt - wyswietla tresc pytania i jego kategorie.
- **Zaznaczenie obszaru** (klik + przeciagniecie) - powiekszenie fragmentu wykresu. Podwojne klikniecie wraca do widoku calosciowego.
- **Klikniecie kategorii w legendzie** - ukrywa/pokazuje dana kategorie. Przydatne, zeby np. zobaczyc tylko Biologie i Wszechswiat jednoczesnie.
- **Podwojne klikniecie kategorii w legendzie** - pokazuje TYLKO ta kategorie (reszta znika). Ponowne podwojne klikniecie przywraca wszystkie.

---

#### Co widac na wykresie - interpretacja wynikow

**1. Pytania dzieci nie tworza scisle oddzielonych grup tematycznych.**

To jest najwazniejszy i jednoczesnie najbardziej interesujacy wynik. Kategorie nadane w PDF (Fizyka, Biologia, Wszechswiat...) nie przekladaja sie na wyrazne, oddzielone "wyspy" na wykresie. Kolory mieszaja sie. Dlaczego?

Bo **dzieci nie mysla kategoriami akademickimi**. Pytanie "Dlaczego niebo jest niebieskie?" jest w PDF w kategorii Fizyka, ale semantycznie jest rownie bliskie Wszechswiatowi. Pytanie "Czemu Ziemia ma jadro jak Slonce?" laczy geologie z astrofizyka. To naturalne - dziecieca ciekawosc nie zna granic dyscyplin i to jest jej sila.

**2. Mimo to pewne grupy sie wyrozniaja.**

- **Biologia** (zielone punkty) tworzy najbardziej zwarta grupe - pytania o zwierzeta ("Dlaczego zebry maja paski?", "Czemu psy maja mokre nosy?") maja charakterystyczne slownictwo, ktore model latwo rozpoznaje.
- **Wszechswiat** (fioletowe) grupuje sie w oddzielnej czesci wykresu - pytania o kosmos, gwiazdy, czarne dziury, planety tworza dosc spojny klaster.
- **Fizyka** (niebieskie) i **Czlowiek** (czerwone) sa bardziej rozproszone - co odzwierciedla szerokosc tych kategorii. Fizyka obejmuje zarowno pytania o atomy, jak i o samoloty czy swiatlo.

**3. Sasiedztwo punktow roznych kolorow ujawnia mosty miedzy dziedzinami.**

Warto przegladac interaktywny wykres i szukac miejsc, gdzie pytania z roznych kategorii sa blisko siebie. Te "mosty" moga byc inspiracja dla prowadzacych zajecia na LAMU - pokazuja, ktore tematy mozna polaczyc w jednym wykladzie czy warsztacie. Na przyklad:
- Pytania o dinozaury (Biologia) sasiaduja z pytaniami o skamieliny i kamienie (Ziemia) - mozna zrobic zajecia "od kosci do skaly".
- Pytania o atomy i czastki (Fizyka) lacza sie z pytaniami o pierwiastki (Chemia) - naturalny most miedzy tymi przedmiotami.
- Pytania o cialo ludzkie (Czlowiek) czesto sa blisko pytan o zwierzeta (Biologia) - dzieci porownuja siebie ze swiatem przyrody.

**4. Kategorie redakcyjne vs. semantyczne - co wynika dla organizacji LAMU.**

Podzial pytan na kategorie w PDF jest redakcyjny - nadany przez czlowieka. Model jezykowy sugeruje, ze niektore pytania moglyby byc przypisane inaczej, a wiele pytan jest z natury interdyscyplinarnych. To moze byc wskazowka dla organizatorow:
- Nie trzeba na sile trzymac sie sztywnych kategorii - jedno zajecie moze laczyc Fizyke z Wszechswiatem albo Biologie z Chemia.
- Pytania, ktore na wykresie sa daleko od swojej grupy, to czesto te najbardziej oryginalne i interdyscyplinarne - warto je wyroznic.
- Gestosc punktow w danym obszarze pokazuje, co dzieci interesuje najbardziej - tam, gdzie jest duzo punktow blisko siebie, jest "gorace" zagadnienie.

**5. Czego nie pokazuje ten wykres.**

- Osie X i Y (UMAP 1, UMAP 2) nie maja znaczenia fizycznego - to abstrakcyjne wymiary po redukcji z 768 wymiarow. Wazne sa tylko wzgledne odleglosci miedzy punktami, nie ich bezwzgledna pozycja.
- Wykres nie mowi nic o trudnosci pytan ani o wieku dzieci - pokazuje wylacznie podobienstwo tresciowe.
- Mala liczba pytan w kategoriach Matematyka, Historia i Chemia sprawia, ze te grupy sa slabo reprezentowane i nie warto wyciagac z nich daleko idacych wnioskow.

## 5. Eksport do HTML (offline)

In [ ]:
HTML_PATH = "lamu_questions_2026.html"
PNG_PATH = "lamu_questions_static.png"

fig.write_html(HTML_PATH, include_plotlyjs=True, full_html=True)
print(f"Zapisano interaktywny wykres: {HTML_PATH}")
print(f"Zapisano statyczny PNG:      {PNG_PATH}")

# Download in Colab
try:
    from google.colab import files
    files.download(HTML_PATH)
    files.download(PNG_PATH)
    print("Pobieranie plikow...")
except ImportError:
    print(f"Otworz plik {HTML_PATH} w przegladarce, aby zobaczyc wykres offline.")

## 6. Nowe pytania - projekcja na istniejacy UMAP

Ponizej dodajemy nowe pytania (spoza oryginalnego zbioru) i projektujemy je na istniejaca mape UMAP za pomoca `reducer.transform()`. Dzieki temu nowe punkty laduja w tej samej przestrzeni co oryginalne pytania i mozna zobaczyc, do ktorych klastrow tematycznych sa najblizsze.

**Wazne:** `transform()` uzywa juz wytrenowanego UMAP - nie zmienia pozycji istniejacych punktow, tylko znajduje najlepsze miejsce dla nowych.

In [ ]:
# --- Nowe pytania do wprojektowania na istniejacy UMAP ---

new_questions = [
    "Czy jak się pije rumianek, to czy da się uspokoić bobasa w brzuchu?",
    "Czemu sny się nie spełniają?",
    "Dlaczego ludzie śmiecą?",
    "Czy z kosmosu mogą przylecieć ufoludki?",
    "Ile jest wszystkich żywych organizmów na całym świecie?",
    "Czemu tylko koale mogą jeść eukaliptus, a inne zwierzęta nie?",
    "Czemu na ziemi rosną drzewa i rośliny, a na innych planetach nie?",
    "Czemu śnieg nie jest kolorowy?",
    "Czemu ludzie nie są trujący?",
    "Czemu jak się obcina końcówki włosów, to wtedy rosną szybciej?",
]

# Embeddingi nowych pytan (ten sam model)
new_embeddings = model.encode(new_questions, show_progress_bar=False, batch_size=32)

# Projekcja na istniejacy UMAP (transform, NIE fit_transform)
new_coords = reducer.transform(new_embeddings)

# DataFrame z nowymi pytaniami
df_new = pd.DataFrame({
    'topic': 'Nowe pytania',
    'question': new_questions,
    'x': new_coords[:, 0],
    'y': new_coords[:, 1],
    'question_short': [q[:80] + '...' if len(q) > 80 else q for q in new_questions]
})

print(f"Wprojektowano {len(df_new)} nowych pytan na istniejacy UMAP")
df_new[['question', 'x', 'y']]

In [ ]:
# --- Wykres: oryginalne pytania + nowe pytania (wyroznionym kolorem) ---

# Polacz oba DataFrame
df_combined = pd.concat([df, df_new], ignore_index=True)

# Dodaj kolor dla nowych pytan
topic_colors_extended = {**topic_colors, 'Nowe pytania': '#FFD700'}  # zloty/zolty

fig2 = px.scatter(
    df_combined, x='x', y='y',
    color='topic',
    hover_data={'question': True, 'topic': True, 'x': False, 'y': False,
                'question_short': False},
    color_discrete_map=topic_colors_extended,
    title='LAMU 2026 - Pytania dzieci + nowe pytania (złote gwiazdki)',
    labels={'topic': 'Kategoria', 'x': 'UMAP 1', 'y': 'UMAP 2'},
    width=900, height=700
)

# Nowe pytania - wieksze, z czarna obwodka, gwiazdka
for trace in fig2.data:
    if trace.name == 'Nowe pytania':
        trace.marker.size = 14
        trace.marker.opacity = 1.0
        trace.marker.symbol = 'star'
        trace.marker.line = dict(width=1.5, color='black')
    else:
        trace.marker.size = 8
        trace.marker.opacity = 0.6
        trace.marker.line = dict(width=0.5, color='white')

fig2.update_traces(
    hovertemplate='<b>%{customdata[1]}</b><br>%{customdata[0]}<extra></extra>'
)

fig2.update_layout(
    legend_title_text='Kategoria',
    font=dict(size=12),
    hoverlabel=dict(font_size=11),
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgray'),
    yaxis=dict(showgrid=True, gridcolor='lightgray')
)

fig2.show()

# Eksport do HTML
HTML_NEW_PATH = "lamu_questions_2026_z_nowymi.html"
fig2.write_html(HTML_NEW_PATH, include_plotlyjs=True, full_html=True)
print(f"\nZapisano: {HTML_NEW_PATH}")

try:
    from google.colab import files
    files.download(HTML_NEW_PATH)
except ImportError:
    pass